In [13]:

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from events_data_core.event_impact_analyzer import analyze_event_impact, EventImpactParams
from events_data_core.plot_utils import plot_price_real
from events_data_core.stock_data_provider import get_stock_data

# ── Параметры анализа ─────────────────────────────────────────────────────
TICKERS = ['LKOH', 'GAZP', 'SBER', 'NVTK']

PARAMS = EventImpactParams(
    baseline_start='2020-07-01',
    baseline_end='2021-12-31',
    window=20,
)

# загружаем данные первого тикера для Dash-визуализации событий
stock_data = get_stock_data(TICKERS[0])
stock_data.head()

,DATE,OPEN,HIGH,LOW,CLOSE,VOL
0,2002-01-08,403.15,426.48,403.15,420.00,1385900.0
1,2002-01-09,369.50,433.35,369.50,426.61,1476737.0
2,2002-01-10,424.10,431.30,420.01,426.00,1566808.0
3,2002-01-11,425.00,431.70,423.00,428.01,808508.0
4,2002-01-14,424.90,424.90,415.50,418.00,824284.0


In [14]:
import duckdb

# подключаемся (in-memory, без файла)
con = duckdb.connect()

# читаем два csv
con.execute("""
            CREATE TABLE event_tags AS
            SELECT *
            FROM read_csv_auto('../data/db/event_tags.csv');
            """)

con.execute("""
            CREATE TABLE events AS
            SELECT *
            FROM read_csv_auto('../data/db/events.csv');
            """)

# извлекаем только санкционные события
query = """
        SELECT e.*
        FROM events e
                 JOIN event_tags t ON e.id = t.event_id
        WHERE t.tag_code = 'SANCTIONS'
          and e.date_start > '2022-01-01'
        """

sanctions_df = con.execute(query).df()
sanctions_df

,id,date_start,date_end,event
0,017f1eba-7c00-4bed-8a0c-e5ee2c76f009,2022-02-21,2022-02-23,Принятие первого пакета санкций против России.
1,017f2b9a-7200-4fca-a35a-c68140153c96,2022-02-24,2022-02-25,Принятие второго пакета санкций против России.
2,017f5c86-fc00-47eb-8988-aeb8b647cc14,2022-02-26,2022-03-14,Принятие третьего пакета санкций против России.
3,017fbe5f-7000-4aa7-b4d4-3688dbe73a96,2022-03-15,2022-04-04,Принятие четвертого пакета санкций против России.
4,01808c5d-f000-4f60-85eb-0aa34a373e1a,2022-04-05,2022-06-02,Принятие пятого пакета санкций против России.
5,01819302-7400-4bdb-8219-df4aec4b9fc7,2022-06-03,2022-07-15,Принятие шестого пакета санкций против России.
6,0182df2c-7200-4a21-96e5-597baa799d81,2022-07-21,2022-10-04,Принятие седьмого пакета санкций против России.
7,01845ed6-7800-41fd-af56-a56104072a56,2022-10-06,2022-12-15,Принятие 8 пакета санкций против России.
8,0185cc79-fc00-46ca-8a45-fd3c3a0342f8,2022-12-16,2023-02-24,Принят 9 пакет санкций против России.
9,0187b08f-7400-42c0-86b2-cb3b9c32c53a,2023-02-25,2023-06-21,Принятие 10 пакета санкций против России.


In [15]:
from events_data_core.plot_utils import plot_2d_events

app = plot_2d_events(stock_data['DATE'], stock_data['CLOSE'], sanctions_df)
app.run(debug=True, jupyter_mode='inline')

In [16]:
# ── Анализ влияния санкционных пакетов на все тикеры ─────────────────────
neighbor_dates = pd.to_datetime(sanctions_df['date_start']).tolist()

records = []
for ticker in TICKERS:
    for i, (_, row) in enumerate(sanctions_df.iterrows()):
        result = analyze_event_impact(
            ticker=ticker,
            event_date=row['date_start'],
            params=PARAMS,
            event_text=row['event'],
            neighbor_dates=neighbor_dates,
        )
        if result is None:
            continue
        records.append({
            'Тикер':                  ticker,
            'Пакет':                  f"Пакет {i + 1}",
            'Дата':                   result.event_date.date(),
            'Обрезано':               result.clipped,
            'Дней до':                result.days_before,
            'Дней после':             result.days_after,
            'Доходность точечная, %': result.point_return_pct,
            'Доходность средняя, %':  result.avg_return_pct,
            'CAR, %':                 result.car_pct,
            'Волатильность до, %':    result.vol_before_pct,
            'Волатильность после, %': result.vol_after_pct,
            'Коэф. волатильности':    result.vol_ratio,
            'Объём до':               int(result.volume_before),
            'Объём после':            int(result.volume_after),
            'Коэф. объёма':           result.volume_ratio,
        })

results_df = pd.DataFrame(records)
display(results_df)

,Тикер,Пакет,Дата,Обрезано,Дней до,Дней после,"Доходность точечная, %","Доходность средняя, %","CAR, %","Волатильность до, %","Волатильность после, %",Коэф. волатильности,Объём до,Объём после,Коэф. объёма
0,LKOH,Пакет 5,2022-04-05,True,6,19,-12.53,-13.56,-13.55,5.973,4.598,0.77,386273,588187,1.52
1,LKOH,Пакет 6,2022-06-03,True,19,15,-11.99,-5.69,3.81,2.765,2.390,0.86,396116,496259,1.25
2,LKOH,Пакет 7,2022-07-21,True,18,20,-3.51,0.52,6.24,1.654,1.964,1.19,456949,459762,1.01
3,LKOH,Пакет 8,2022-10-06,False,20,20,5.70,4.87,18.26,3.192,1.681,0.53,830156,785829,0.95
4,LKOH,Пакет 9,2022-12-16,False,20,20,-14.33,-11.97,-14.68,0.657,2.747,4.18,405171,474381,1.17
5,LKOH,Пакет 10,2023-02-25,False,20,20,9.43,4.50,9.26,0.818,1.264,1.54,380287,609004,1.60
6,LKOH,Пакет 11,2023-06-23,False,20,20,4.74,0.64,7.43,2.251,1.152,0.51,1309457,943180,0.72
7,LKOH,Пакет 12,2023-12-18,False,20,20,-5.46,-5.17,1.40,1.473,0.712,0.48,712056,484527,0.68
8,LKOH,Пакет 13,2024-02-23,False,20,20,6.64,3.20,2.71,0.823,0.983,1.19,576268,1057611,1.84
9,LKOH,Пакет 14,2024-06-24,False,20,20,-10.82,-5.02,-4.36,1.910,1.716,0.90,1096881,796125,0.73


● Интерпретация колонок таблицы
  ---
  Метрики доходности

  Доходность точечная, %
  Цена в последний день окна "после" vs цена в первый день окна "до".

  Нестабильна — зависит от двух конкретных дней. Используй только для наглядности.

  Доходность средняя, %
  Средняя цена за 20 дней после события vs средняя цена за 20 дней до.

  Надёжнее точечной — сглаживает случайные дни. Основная метрика доходности.

  - +5% → акция в среднем торговалась на 5% выше после события
  - -15% → рынок в среднем упал на 15% в окне после события

  ---
  CAR (Cumulative Abnormal Return), %

  Сумма аномальных дневных доходностей в окне "после".
  Аномальная = фактическая − ожидаемая (baseline 2020–2021).

  Самая важная метрика. Отвечает на вопрос: акция упала из-за события, или она и так бы упала?

  - CAR = -15% → акция потеряла 15% сверх того, что было бы без события
  - CAR = +0% при доходность средняя = -5% → рынок падал и без события, само событие не добавило эффекта
  - CAR = -5% при доходность средняя = -15% → из 15% падения только 5% объясняется аномалией, остальные 10% — норма рынка того периода

  ---
  Волатильность

  Волатильность до/после, % — среднеквадратичное отклонение дневных доходностей в окне (в %).

  Коэф. волатильности = после / до
  - > 1 → рынок нервничал после события (неопределённость выросла)
  - < 1 → рынок успокоился
  - 4.18 у LKOH на пакет 9 → колебания выросли в 4 раза — явный сигнал стресса

  ---
  Объём

  Объём до/после — средний дневной объём торгов в акциях.

  Коэф. объёма = после / до
  - > 1 → объём вырос, рынок реагировал активно (инвесторы перекладывались)
  - < 1 → объём упал, реакции почти не было
  - 1.84 → объём вырос почти в 2 раза — событие привлекло внимание

  ---
  Как читать строку в целом

  Пример: SBER, начало войны
  avg = -50.5%,  CAR = -16.7%,  vol_ratio = 0.79,  vol_ratio = 0.79
  → Акция упала на 50.5% в среднем за 20 дней после.
  → Из них ~17% — именно аномальная реакция на событие, остальные ~33% — рыночный контекст того периода (биржа была закрыта, торги заморожены).
  → Волатильность снизилась — парадоксально, но объясняется заморозкой торгов: дни без торгов дают нулевые доходности, сжимая std.


In [17]:
# ── Барчарты для выбранного тикера ───────────────────────────────────────
PLOT_TICKER = 'LKOH'  # поменяй на любой из TICKERS

df = results_df[results_df['Тикер'] == PLOT_TICKER]
labels     = df['Пакет']
colors_ret = ['green' if v >= 0 else 'red' for v in df['Доходность средняя, %']]
colors_car = ['green' if v >= 0 else 'red' for v in df['CAR, %']]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Средняя доходность, %',
        'CAR (накопл. аномальная доходность), %',
        'Коэффициент волатильности (после / до)',
        'Коэффициент объёма торгов (после / до)',
    ]
)

fig.add_trace(go.Bar(x=labels, y=df['Доходность средняя, %'],  marker_color=colors_ret,   name='Доходность'),    row=1, col=1)
fig.add_trace(go.Bar(x=labels, y=df['CAR, %'],                  marker_color=colors_car,   name='CAR'),           row=1, col=2)
fig.add_trace(go.Bar(x=labels, y=df['Коэф. волатильности'],     marker_color='steelblue',  name='Волатильность'), row=2, col=1)
fig.add_trace(go.Bar(x=labels, y=df['Коэф. объёма'],            marker_color='darkorange', name='Объём'),         row=2, col=2)

fig.add_hline(y=1, row=2, col=1, line_dash='dash', line_color='gray', line_width=1)
fig.add_hline(y=1, row=2, col=2, line_dash='dash', line_color='gray', line_width=1)

fig.update_layout(
    height=750,
    title_text=(
        f'Влияние пакетов санкций на акции {PLOT_TICKER}  |  window={PARAMS.window} дн.  |  '
        f'baseline={PARAMS.baseline_start}–{PARAMS.baseline_end}'
    ),
    showlegend=False,
    template='plotly_white',
)

fig

In [18]:
# ── Нормализованный график цены с поправкой на инфляцию ──────────────────
NORM_TICKER = 'LKOH'  # поменяй на любой из TICKERS

stock_data_norm = get_stock_data(NORM_TICKER)

fig2 = plot_price_real(
    stock_data_norm,
    normalize_date='2022-02-24',
    events_df=sanctions_df,
    title=f'Реальная цена {NORM_TICKER} (с поправкой на инфляцию), нормализована к 24.02.2022 = 100',
)

fig2